# Weaviate Quickstart with Local Instance

This notebook demonstrates how to connect to a local Weaviate instance, define a collection, import data, and perform searches.

In [ ]:
!pip install -U weaviate-client requests
import weaviate
import json
import requests

In [ ]:
# Connect to local Weaviate
client = weaviate.connect_to_local(port=8080)
print("Connected to Weaviate:", client.is_ready())

In [ ]:
# Define a collection
if client.collections.exists("Question"):
    client.collections.delete("Question")

questions = client.collections.create(
    name="Question",
    vectorizer_config=weaviate.classes.config.Configure.Vectorizer.text2vec_contextionary()
)

In [ ]:
# Import data
resp = requests.get('https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json')
data = json.loads(resp.text)

with questions.batch.fixed_size(batch_size=100) as batch:
    for i, d in enumerate(data):
        properties = {
            "answer": d["Answer"],
            "question": d["Question"],
            "category": d["Category"],
        }
        batch.add_object(properties=properties)
print("Imported objects")

In [ ]:
# Search
response = questions.query.near_text(query="biology", limit=2)
for o in response.objects:
    print(o.properties)